# Comparing two REV runs on different meshes: `twillA` against `tensorA`

**What it does:** scores the twill run (job 3113) against the plain-weave reference `tensorA` (job 3107) on the
predictions T1–T6 of `REV_pipeline/README.md` ("Weave test: twill 1/3"), written before the run.


 Here:
1. every quantity is a sum, a mean or an extreme over a set of cells that is defined the same way on each mesh (yarn cells,
   the yarns along y, free cells of the fabric box, the first cell row above the bath);
2. everything is in physical units (mm, mm³, mm²/√s), each mesh with its own cell volumes. Never cell by cell.

**Plan:**
1. Read each mesh and its fixed fields: what each mesh contains.
2. Quantities at every write, with the same definitions on both meshes; check against the solver's own water budget.
3. Scoring T1–T6: fits, time windows, the solver log, plots (next step).

## 1. Read the runs
### 1.1 Paths and settings

In [ ]:
import csv
import os
import re
import subprocess
import time

import numpy as np

RESULTS = os.path.expanduser("~/PDM_results")
REF, NEW = "tensorA", "twillA"                      # reference run, run to score
RUNS = {
    "tensorA": f"{RESULTS}/REV_pipeline_tensorA",   # plain weave, job 3107, 30 ms
    "twillA":  f"{RESULTS}/REV_pipeline_twillA",    # twill 1/3, job 3113, 20 ms
}



### 1.2 What an OpenFOAM field file looks like
Each time folder (`0.01` = 10 ms) holds one file per field. A file has a header (`FoamFile {...}`), `dimensions`, then
`internalField` (the value in every cell), then `boundaryField` (the values on the patches; not used here).
`internalField` is either `uniform 0.5;` (the same value in every cell) or `nonuniform List<scalar>`, followed by the
number of cells and the values in parentheses, one per line (vectors: `(x y z)` per line).
**The values are in the cell order of the mesh** (`constant/polyMesh`). So in one run, `alpha[i]`, `eps[i]` and
`C[i]` (cell centre) belong to the same cell `i`, at every time. The cell below prints the start and the end of the list.

In [2]:
path = f"{RUNS[NEW]}/0.01/alpha.wetting"
lines = open(path).readlines()
print("".join(lines[:26]))
i = next(k for k, l in enumerate(lines) if l.startswith("boundaryField"))
print("...\n" + "".join(lines[i - 4:i + 6]))
print(f"({len(lines):,} lines in the file)")

/*--------------------------------*- C++ -*----------------------------------*\
  =========                 |
  \\      /  F ield         | OpenFOAM: The Open Source CFD Toolbox
   \\    /   O peration     | Website:  https://openfoam.org
    \\  /    A nd           | Version:  9
     \\/     M anipulation  |
\*---------------------------------------------------------------------------*/
FoamFile
{
    format      ascii;
    class       volScalarField;
    location    "0.01";
    object      alpha.wetting;
}
// * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * //

dimensions      [0 0 0 0 0 0 0];

internalField   nonuniform List<scalar> 
254163
(
0.000481577
0.000388155
0.000252202
0.000129999
5.51723e-05

...
3.52094e-60
)
;

boundaryField
{
    inlet
    {
        type            inletOutlet;
        inletValue      uniform 1;

(279,810 lines in the file)


The reader keeps only `internalField` and returns a numpy array with one entry per cell. (The older
`analyseCut.field` does the same, more slowly.)

In [3]:
def read_field(path, n=None, vector=False):
    """internalField of an ASCII OpenFOAM field file as a numpy array, one entry per cell: shape (n,) for a scalar,
    (n, 3) for a vector. n is needed only for a 'uniform' field (it gives no cell count)."""
    txt = open(path).read()
    body = txt.split("internalField", 1)[1]               # everything after the keyword
    if body.lstrip().startswith("uniform"):               # 'uniform 0.5;' or 'uniform (0 0 0);'
        value = body.split("uniform", 1)[1].split(";", 1)[0]
        v = np.array(value.replace("(", " ").replace(")", " ").split(), float)
        return np.tile(v, (n, 1)) if vector else np.full(n, v[0])
    head, rest = body.split("(", 1)                       # head = ' nonuniform List<scalar> \n254163\n'
    count = int(head.split()[-1])
    if vector:                                            # lines '(x y z)'; the list ends with ')' alone on a line
        values = rest.split("\n)", 1)[0].replace("(", " ").replace(")", " ")
        a = np.array(values.split(), float).reshape(-1, 3)
    else:                                                 # numbers; the list ends at the first ')'
        a = np.array(rest.split(")", 1)[0].split(), float)
    assert len(a) == count, f"{path}: read {len(a)} values, the file says {count}"
    return a

In [4]:
a = read_field(f"{RUNS[NEW]}/0.01/alpha.wetting")
print(a.shape, a[:5], a.min(), a.max())

(254163,) [4.81577e-04 3.88155e-04 2.52202e-04 1.29999e-04 5.51723e-05] -9.46873e-21 0.991978


### 1.3 The fixed fields of each mesh
Read once per run from the `0` folder:
- `C`: cell centres [m]; `V`: cell volumes [m³]. They are written into the local copies by
  `postProcess -func writeCellCentres` / `writeCellVolumes` (the cell runs it when a file is missing; that needs OpenFOAM
  in the environment, otherwise run it once in a terminal);
- `eps`: porosity, 0.5 in yarn cells and 1 in free cells → **porous mask `P = eps <= 0.99`**, as in the solver and every
  script;
- `yarnID`: the yarn a cell belongs to (−1: none). It also marks free cells that a yarn surface touches, so always use it
  together with `P`;
- the vertices of `system/blockMeshDict`: the domain bounds and, in z, the planes between the padding layers and the
  fabric box.

The masks are boolean arrays with one entry per cell: `P` porous, `F` free, `box` (z between the padding layers),
`row0` (first cell row above the bath plane). A sum over a mask, e.g. `(a * V)[F & box].sum()`, is "the water in the free
cells of the box".

In [5]:
def block_vertices(run):
    """sorted distinct x, y and z of the blockMesh vertices: the domain bounds and, in z, the planes between
    the padding layers and the fabric box"""
    txt = open(f"{run}/system/blockMeshDict").read()
    v = txt.split("vertices", 1)[1].split(");", 1)[0]
    pts = np.array(re.findall(r"\(\s*([-\d.e+]+)\s+([-\d.e+]+)\s+([-\d.e+]+)\s*\)", v), float)
    return [np.unique(pts[:, i]) for i in range(3)]


def load_mesh(run):
    for name, func in (("C", "writeCellCentres"), ("V", "writeCellVolumes")):
        if not os.path.isfile(f"{run}/0/{name}"):
            subprocess.run(["postProcess", "-func", func, "-time", "0"], cwd=run, check=True,
                           stdout=subprocess.DEVNULL)
    C = read_field(f"{run}/0/C", vector=True)
    n = len(C)
    m = dict(run=run, n=n, C=C, V=read_field(f"{run}/0/V", n), eps=read_field(f"{run}/0/eps", n),
             yid=read_field(f"{run}/0/yarnID", n).astype(int))
    xv, yv, zv = block_vertices(run)
    m["width"] = xv[-1] - xv[0]                          # fabric width, across the wicking direction [m]
    m["y0"], m["y1"] = yv[0], yv[-1]                     # bath plane, closed end [m]
    m["zbox"] = (zv[1], zv[2])                           # fabric box between the padding layers [m]
    m["P"] = m["eps"] <= 0.99                            # porous (yarn) cells
    m["F"] = ~m["P"]                                     # free cells
    m["box"] = (C[:, 2] > zv[1]) & (C[:, 2] < zv[2])
    half_dy = C[:, 1].min() - yv[0]                      # first cell centre = bath plane + dy/2
    m["row0"] = C[:, 1] < yv[0] + 2 * half_dy            # first cell row above the bath plane
    m["d"] = [np.median(np.diff(np.unique(np.round(C[:, i], 9)))) for i in range(3)]   # cell size dx, dy, dz [m]
    return m


M = {name: load_mesh(run) for name, run in RUNS.items()}

for name, m in M.items():
    V, P, F, box = m["V"], m["P"], m["F"], m["box"]
    print(f"{name}: {m['n']:,} cells ({P.sum():,} porous, {F.sum():,} free)")
    print("   cell {:.2f} x {:.2f} x {:.2f} um, volume {:.4e} m3 (min/max {:.6f})".format(
        *(np.array(m["d"]) * 1e6), V.mean(), V.min() / V.max()))
    print(f"   width {m['width'] * 1e3:.4f} mm, bath plane y {m['y0'] * 1e3:.4f} mm, closed end {m['y1'] * 1e3:.4f} mm, "
          f"fabric box z {m['zbox'][0] * 1e3:.4f} to {m['zbox'][1] * 1e3:.4f} mm "
          f"({(m['zbox'][1] - m['zbox'][0]) * 1e3:.4f} mm thick)")
    print(f"   yarn volume {V[P].sum() * 1e9:.4f} mm3 (pore space {(m['eps'] * V)[P].sum() * 1e9:.4f} mm3), "
          f"free volume in the box {V[F & box].sum() * 1e9:.4f} mm3")
    print(f"   first fabric row: {(F & box & m['row0']).sum()} free cells, {(P & m['row0']).sum()} porous cells")

tensorA: 242,060 cells (78,489 porous, 163,571 free)
   cell 19.98 x 20.05 x 20.14 um, volume 8.0693e-15 m3 (min/max 1.000000)
   width 1.8182 mm, bath plane y 0.3333 mm, closed end 3.0000 mm, fabric box z -0.0705 to 0.2115 mm (0.2820 mm thick)
   yarn volume 0.6333 mm3 (pore space 0.3167 mm3), free volume in the box 0.7339 mm3
   first fabric row: 944 free cells, 330 porous cells
twillA: 254,163 cells (75,936 porous, 178,227 free)
   cell 19.98 x 20.05 x 20.50 um, volume 8.2129e-15 m3 (min/max 1.000000)
   width 1.8182 mm, bath plane y 0.3333 mm, closed end 3.0000 mm, fabric box z -0.0804 to 0.2272 mm (0.3075 mm thick)
   yarn volume 0.6237 mm3 (pore space 0.3118 mm3), free volume in the box 0.8674 mm3
   first fabric row: 1043 free cells, 322 porous cells


**What the two meshes have in common, and what differs:**
- x and y are the same (cells 19.98 × 20.05 µm, width 1.818 mm, bath plane at y 0.333 mm, closed end at 3.0 mm): same
  yarn densities, same construction of the mid-gap bath.
- z differs: the twill box is thicker (0.3075 against 0.282 mm), 15 layers of 20.50 µm against 14 of 20.14 µm, so **one
  cell is 1.8 % larger in the twill**. A count of "cells of water" would be biased by 1.8 % before any physics; hence mm³.
- The twill has 1.5 % less yarn volume (0.6237 against 0.6333 mm³: the ×0.985 in the prediction of T3) and 18 % more
  free volume in the box (thicker box).
- The first fabric row has 1,043 free cells in the twill against 944 (the "of 1,043" of T5).

### 1.4 Which way each yarn runs
Assumption of every definition below: yarns 0–4 run along y (the wicking direction) and 6–9 cross them, in both meshes.
Checked here from the extent of each yarn's porous cells, not assumed.

In [6]:
for name, m in M.items():
    print(name)
    along = []
    crossing = []
    ymid ={}
    for k in sorted(set(m["yid"][m["P"]])):
        c = m["C"][m["P"] & (m["yid"] == k)]
        ext = (c.max(axis=0) - c.min(axis=0)) * 1e3
        if ext[1]>ext[0]:
            along.append(int(k))
        else: 
            crossing.append(int(k))
            ymid[k]=(c[:, 1].mean() - m["y0"]) * 1e3 
        print(f"   yarn {k}: {len(c):6,} porous cells, extent x {ext[0]:.3f}  y {ext[1]:.3f}  z {ext[2]:.3f} mm"
              f"  -> along {'y' if ext[1] > ext[0] else 'x'}")
    m["along"]=along 
    m["crossing"]=sorted(crossing, key=ymid.get) 
    print (f" yarns along y: {m['along']}, crossing: {m['crossing']}")
    for i, k in enumerate(m["crossing"], start=1):
        print(f"   X{i} = yarn {k}, {ymid[k]:.3f} mm above the bath plane")

tensorA
   yarn 0:  5,471 porous cells, extent x 0.140  y 2.647  z 0.242 mm  -> along y
   yarn 1: 10,856 porous cells, extent x 0.300  y 2.647  z 0.242 mm  -> along y
   yarn 2: 10,756 porous cells, extent x 0.280  y 2.647  z 0.242 mm  -> along y
   yarn 3: 10,856 porous cells, extent x 0.300  y 2.647  z 0.242 mm  -> along y
   yarn 4:  5,471 porous cells, extent x 0.140  y 2.647  z 0.242 mm  -> along y
   yarn 6:  8,832 porous cells, extent x 1.798  y 0.321  z 0.262 mm  -> along x
   yarn 7:  8,715 porous cells, extent x 1.798  y 0.321  z 0.262 mm  -> along x
   yarn 8:  8,705 porous cells, extent x 1.798  y 0.321  z 0.262 mm  -> along x
   yarn 9:  8,827 porous cells, extent x 1.798  y 0.321  z 0.262 mm  -> along x
 yarns along y: [0, 1, 2, 3, 4], crossing: [6, 7, 8, 9]
   X1 = yarn 6, 0.333 mm above the bath plane
   X2 = yarn 7, 1.001 mm above the bath plane
   X3 = yarn 8, 1.666 mm above the bath plane
   X4 = yarn 9, 2.333 mm above the bath plane
twillA
   yarn 0:  5,329 porous 

As assumed in both meshes: 0–4 along y (0 and 4 are the halves at the x-periodic planes, 0.14 mm wide), 6–9 along x over
the full width.

## 2. Quantities at every write
The same definitions on both meshes (volumes in mm³, lengths in mm). In a porous cell α is the saturation S (water
fraction of the pore space), so the water volume of a cell is `eps·α·V`. In a free cell α is the VOF water fraction and the
water volume is `α·V`.

| quantity | definition | for |
|---|---|---|
| `front` [mm] | largest y of a porous cell of yarns 0–4 with α > 0.1 (absolute y, as `compareTensor.py`: bath plane 0.333 mm, closed end 3.0 mm) | T2 |
| `w_yarn` [mm³] | Σ eps·α·V over the porous cells | T3 split |
| `w_gap` [mm³] | Σ α·V over the free cells of the fabric box | T5 |
| `w_pad` [mm³] | Σ α·V over the free cells of the padding layers | T1 (leak) |
| `uptake` [mm²] | (w_yarn + w_gap + w_pad) / fabric width: the water above the bath plane per unit width of fabric | T3 |
| `S6` … `S9` | mean α of the porous cells of crossing yarn k | T4 |
| `row0_wet` | free box cells of the first row with α > 0.5 (the gaps that still touch the bath) | T5 |
| `U_free`, `U_por` [m/s], `a_min`, `a_max` | extremes over free / porous cells, over all cells | T1 |

**Why per unit width:** a strip test weighs the water per cm of strip width. The twill box is thicker than the plain
weave box (see 1.3), so a per-cross-section uptake would mix the weave effect with the thickness.

In [27]:
def writes(run, t_max):
    """time folders with an alpha.wetting file, up to t_max [ms] (folder names, sorted by time)"""
    ts = [x for x in os.listdir(run) if re.fullmatch(r"[0-9.e+-]+", x)
          and os.path.isfile(f"{run}/{x}/alpha.wetting") and float(x) * 1e3 <= t_max + 1e-9]
    return sorted(ts, key=float)


def per_write(m, t):
    """the quantities of one write of one run (table above)"""
    a = read_field(f"{m['run']}/{t}/alpha.wetting", m["n"])
    U = read_field(f"{m['run']}/{t}/U", m["n"], vector=True)
    Umag = np.linalg.norm(U, axis=1)
    C, V, P, F, box, yid = m["C"], m["V"], m["P"], m["F"], m["box"], m["yid"]
    wet_y = P & np.isin(yid, m["along"]) & (a > 0.1)          # wet cells of the yarns along y
    r = dict(t=float(t) * 1e3,
             front=C[wet_y, 1].max() * 1e3,
             w_yarn=(m["eps"] * a * V)[P].sum() * 1e9,
             w_gap=(a * V)[F & box].sum() * 1e9,
             w_pad=(a * V)[F & ~box].sum() * 1e9,
             row0_wet=int((F & box & m["row0"] & (a > 0.5)).sum()),
             U_free=Umag[F].max(), U_por=Umag[P].max(), a_min=a.min(), a_max=a.max())
    r["w_total"] = r["w_yarn"] + r["w_gap"] + r["w_pad"]
    r["uptake"] = r["w_total"] / (m["width"] * 1e3)                 # mm3 / mm = mm2
    r["h"] = r["front"] - m["y0"] * 1e3 
    for i, k in enumerate(m["crossing"], start=1):
        r[f"X{i}"] = a[P & (yid == k)].mean()
    return r

end_times = []
for runs in RUNS: 
    T_steps = writes(RUNS[runs],1e9)
    T_sup = float(T_steps[-1])*1e3
    end_times.append(T_sup)
    
T_MAX = min(end_times)



R = {}     # R[run][quantity] = numpy array over the writes
for name, m in M.items():
    t0 = time.time()
    rows = [per_write(m, t) for t in writes(m["run"], T_MAX)]
    R[name] = {k: np.array([r[k] for r in rows]) for k in rows[0]}
    print(f"{name}: {len(rows)} writes (0 to {R[name]['t'][-1]:.1f} ms) read in {time.time() - t0:.0f} s")

tensorA: 41 writes (0 to 20.0 ms) read in 13 s
twillA: 41 writes (0 to 20.0 ms) read in 16 s


In [8]:
def water_budget(run):
    """waterBudget.csv as {time [ms]: row}; a restart repeats a time, the last row is kept"""
    out = {}
    for row in csv.DictReader(open(f"{run}/waterBudget.csv")):
        out[round(float(row["time"]) * 1e3, 6)] = {k: float(v) for k, v in row.items()}
    return out


for name, m in M.items():
    B = water_budget(m["run"])
    s = R[name]
    w_solver = np.array([B[round(t, 6)]["water_m3"] * 1e9 for t in s["t"]])
    rel = np.abs(s["w_total"] / w_solver - 1)
    last = B[round(s["t"][-1], 6)]
    lost = -(last["in_top"] + last["in_bottom"]) * 1e9                # in_ = inflow, so minus = lost
    print(f"{name}: fields / solver water, largest relative difference {rel.max():.1e} (at {s['t'][rel.argmax()]:.1f} ms)")
    print(f"   at {s['t'][-1]:.1f} ms: imbalance {last['imbalance']:.2e} m3, lost through top/bottom {lost:.2e} mm3"
          f" = {lost * 1e-9 / m['V'].mean():.1f} cell volumes of this mesh")

tensorA: fields / solver water, largest relative difference 4.3e-06 (at 9.5 ms)
   at 20.0 ms: imbalance -1.89e-18 m3, lost through top/bottom 3.97e-05 mm3 = 4.9 cell volumes of this mesh
twillA: fields / solver water, largest relative difference 5.1e-06 (at 7.5 ms)
   at 20.0 ms: imbalance -8.50e-18 m3, lost through top/bottom 1.18e-04 mm3 = 14.4 cell volumes of this mesh


Expected: a relative difference of a few 1e-6 (the 6-digit write precision). Anything larger means a wrong mask, a wrong
volume or a misread file. The imbalance should be ~1e-17 m³ or less (`tensorA` had a one-time −8.5e-18 m³ in the first
0.5 ms, as job 3068).

### 2.2 Side by side
Both runs at the same times, and their ratio (twill / plain).

In [9]:
COLS = [("front", "front", "{:7.3f}"), ("uptake", "uptake", "{:7.4f}"), ("w_yarn", "w_yarn", "{:7.4f}"),
        ("w_gap", "w_gap", "{:7.4f}"), ("X1", "X1", "{:6.3f}"), ("X2", "X2", "{:6.3f}"), ("X3", "X3", "{:6.3f}"),
        ("row0_wet", "row0", "{:5.0f}"), ("U_free", "|U|free", "{:7.2f}"), ("a_max", "a_max", "{:7.4f}")]


def index_at(s, t):
    i = np.nonzero(np.isclose(s["t"], t))[0]
    return i[0] if i.size else None


print("t [ms]  run      " + " ".join(f"{h:>{len(f.format(0))}}" for _, h, f in COLS))
for t in (0.5, 1, 2, 5, 10, 15, 20):
    i, j = index_at(R[REF], t), index_at(R[NEW], t)
    if i is None or j is None:
        continue
    for label, s, k in ((REF, R[REF], i), (NEW, R[NEW], j)):
        print(f"{t:6.1f}  {label:8s} " + " ".join(f.format(s[q][k]) for q, _, f in COLS))
    print(f"{'':6s}  {'ratio':8s} " + " ".join(
        f"{R[NEW][q][j] / R[REF][q][i]:{len(f.format(0))}.3f}" if R[REF][q][i] else " " * len(f.format(0))
        for q, _, f in COLS))

t [ms]  run        front  uptake  w_yarn   w_gap     X1     X2     X3  row0 |U|free   a_max
   0.5  tensorA    0.885  0.0197  0.0312  0.0046  0.267  0.002  0.002    79    3.03  0.9936
   0.5  twillA     0.925  0.0200  0.0318  0.0046  0.233  0.002  0.002    70    3.85  0.9958
        ratio      1.045   1.018   1.019   1.011  0.872  1.000  1.000 0.886   1.271   1.002
   1.0  tensorA    1.005  0.0234  0.0401  0.0024  0.412  0.002  0.002    24    2.87  0.9913
   1.0  twillA     1.065  0.0232  0.0401  0.0021  0.345  0.002  0.002    10    4.00  0.9921
        ratio      1.060   0.993   1.000   0.873  0.838  1.000  1.000 0.417   1.391   1.001
   2.0  tensorA    1.145  0.0309  0.0549  0.0012  0.635  0.002  0.002    19    1.23  0.9886
   2.0  twillA     1.266  0.0301  0.0537  0.0010  0.527  0.003  0.002     9    3.46  0.9890
        ratio      1.105   0.975   0.978   0.821  0.830  1.278  1.000 0.474   2.813   1.000
   5.0  tensorA    1.486  0.0469  0.0843  0.0011  0.890  0.092  0.002    19    0

**Check of the definitions:** the `tensorA` rows reproduce the values that the README quotes from `compareTensor.py`
(front 1.005 / 1.486 / 1.847 / 2.409 mm at 1 / 5 / 10 / 20 ms, S6 0.890 at 5 ms, S7 0.450 at 10 ms, S8 0.255 at 20 ms).
The two scripts measure the same things; this one also works across meshes.

## 3. Scoring T1–T6 (next step, together)
- T1 stability: from 2.1 and the table (α max, |U| free after 2 ms), plus stopGuard from the log;
- T2 front: fit front = k√t + c over 2–20 ms, and h² = k²(t + t0) (h from the bath plane); the ratio of k to `tensorA`;
- T3 uptake per width per window: (uptake(t2) − uptake(t1)) / (√t2 − √t1) over 2–5, 5–10, 10–15, 15–20 ms;
- T4 crossing yarns: S6 at 5 ms, S7 at 10 ms, S8 at 20 ms;
- T5 gaps: gap water at 1 ms and after, first-row wet cells;
- T6 dt and cost from the sampled solver log;
- plots: front and uptake against √t, crossing-yarn S against t.

In [ ]:
#T3 script 
start_time=5 
window_width= 5
WINDOWS= range(start_time,T_MAX+1, window_width)
WINDOWS = [(2, 5), (5, 10), (10, 15), (15, 20)] #(ms)

def compute_coeff(s, t1, t2):
    i1 = index_at(s, t1)
    i2 = index_at(s, t2)
    diff_uptake = (s["uptake"][i2])-(s["uptake"][i1])
    dsqrt = np.sqrt(t2* 1e-3)-np.sqrt(t1* 1e-3)
    return diff_uptake/dsqrt 

for t1, t2 in WINDOWS : 
    rref= compute_coeff(R[REF],t1,t2)
    rnew = compute_coeff(R[NEW],t1,t2)
    ratio = rnew/rref 

    if t1 < 5.0 : 
        print(f"Ratio not computed between {t1} and {t2} ms")
    elif 1.0 <=ratio<= 1.05 : 
        print(f"Water uptake inside predicted range between {t1} and {t2} ms ratio = {ratio:.2f}")
    else : 
        print(f"Out of predicted range between {t1} and {t2} ms: ratio = {ratio:.2f}")



Ratio not computed between 2 and 5 ms
Water uptake inside predicted range between 5 and 10 ms ratio = 1.03
Out of predicted range between 10 and 15 ms: ratio = 1.00
Water uptake inside predicted range between 15 and 20 ms ratio = 1.01


In [32]:
print(list(range(5,21,5)))

[5, 10, 15, 20]
